# TP2 - Módulo 2
## Notebook 06 - Predicción Final

**Objetivo:** Cargar el modelo entrenado, aplicar el mismo pipeline de preprocesamiento al dataset sin etiquetar y generar las predicciones finales en una columna `smoking_prediction`.

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Cargar modelo y artefactos del pipeline

In [2]:
model     = joblib.load('../models/xgb_model.joblib')
threshold = joblib.load('../models/threshold.joblib')
feat_names = joblib.load('../models/feature_names.joblib')

print(f'Modelo cargado | Umbral: {threshold:.2f}')
print(f'Features esperadas: {len(feat_names)}')

Modelo cargado | Umbral: 0.48
Features esperadas: 39


## 2. Cargar datos sin etiquetar y aplicar el mismo pipeline

In [3]:
df_raw = pd.read_excel('../data/raw/dataset_sin_etiquetar.xlsx')
print('Dataset sin etiquetar (raw):', df_raw.shape)
df_raw.head()

Dataset sin etiquetar (raw): (5692, 26)


,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar
0,27358,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,...,3.041667,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,Y
1,27364,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,...,4.208333,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,N
2,27368,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,...,2.041667,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,Y
3,27378,M,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,...,3.708333,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,N
4,27381,M,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,...,6.625000,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,Y


In [4]:
# =============================================
# MISMO PIPELINE DE PREPROCESAMIENTO
# (idéntico al usado en el entrenamiento)
# =============================================

def preprocess(df):
    df = df.copy()

    # Eliminar ID
    df = df.drop(columns=['ID'], errors='ignore')

    # Feature Engineering
    df['BMI'] = df['weight(kg)'] / (df['height(cm)'] / 100) ** 2
    df['pulse_pressure'] = df['systolic'] - df['relaxation']
    df['HDL_ratio'] = df['HDL'] / (df['Cholesterol'] + 1e-6)

    for col in ['triglyceride', 'Gtp', 'ALT', 'AST', 'serum creatinine',
                'fasting blood sugar', 'LDL', 'Cholesterol']:
        df[f'log_{col}'] = np.log1p(df[col])

    df['age_group'] = pd.cut(df['age'], bins=[0, 35, 45, 55, 65, 100],
                              labels=[0, 1, 2, 3, 4]).astype(int)

    df['gender_num'] = (df['gender'] == 'M').astype(int)
    df['gender_hemo'] = df['gender_num'] * df['hemoglobin']

    df['gender'] = df['gender'].map({'M': 1, 'F': 0}).astype(int)
    df['oral']   = df['oral'].map({'Y': 1, 'N': 0}).astype(int)
    df['tartar'] = df['tartar'].map({'Y': 1, 'N': 0}).astype(int)

    return df


df_proc = preprocess(df_raw)

# Asegurar que las features estén en el mismo orden
df_proc = df_proc[feat_names]

print('Dataset procesado:', df_proc.shape)
df_proc.head()

Dataset procesado: (5692, 39)


,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,log_Gtp,log_ALT,log_AST,log_serum creatinine,log_fasting blood sugar,log_LDL,log_Cholesterol,age_group,gender_num,gender_hemo
0,1,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,...,0.535518,0.535518,0.559616,0.006231,1.609438,1.396657,1.865371,0,1,0.631250
1,1,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,...,0.847298,0.753772,0.583146,0.040822,1.642228,1.650260,2.187922,0,1,0.587500
2,1,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,...,1.098612,0.829279,0.733969,0.006231,1.625967,1.112406,2.074220,2,1,0.625694
3,1,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,...,0.377294,0.459532,0.606136,0.042820,1.558145,1.549334,2.014903,0,1,0.627083
4,1,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,...,1.084626,0.965081,0.810930,0.040822,1.642228,2.031432,2.438717,0,1,0.670833


## 3. Generar predicciones

In [5]:
# Probabilidades
proba = model.predict_proba(df_proc)[:, 1]

# Predicciones con umbral óptimo
predicciones = (proba >= threshold).astype(int)

print(f'Predicciones generadas con umbral = {threshold:.2f}')
print(f'Total filas: {len(predicciones)}')
print(f'Fumadores predichos (1): {predicciones.sum()} ({predicciones.mean()*100:.1f}%)')
print(f'No fumadores predichos (0): {(predicciones==0).sum()} ({(1-predicciones.mean())*100:.1f}%)')

Predicciones generadas con umbral = 0.48
Total filas: 5692
Fumadores predichos (1): 2826 (49.6%)
No fumadores predichos (0): 2866 (50.4%)


## 4. Exportar resultado con columna `smoking_prediction`

In [6]:
# Agregar predicciones al dataset original
df_resultado = df_raw.copy()
df_resultado['smoking_prediction'] = predicciones

# Verificar que la columna está correcta (solo 0 y 1)
print('Valores únicos en smoking_prediction:', df_resultado['smoking_prediction'].unique())
print('Tipos:', df_resultado['smoking_prediction'].dtype)

df_resultado.head(10)

Valores únicos en smoking_prediction: [1 0]
Tipos: int64


,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar,smoking_prediction
0,27358,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,...,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,Y,1
1,27364,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,...,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,N,1
2,27368,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,...,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,Y,1
3,27378,M,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,...,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,N,0
4,27381,M,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,...,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,Y,1
5,27386,M,45,175,65,3.334722,0.045139,0.045139,0.041667,0.041667,...,0.708333,0.041667,0.041667,0.708333,0.500000,0.791667,Y,0,N,1
6,27388,M,55,160,65,3.500000,0.043056,0.045139,0.041667,0.041667,...,0.543056,0.041667,0.043056,1.541667,1.791667,1.958333,Y,0,Y,0
7,27393,F,50,150,55,3.083333,0.045139,0.045139,0.041667,0.041667,...,0.502083,0.041667,0.004861,0.791667,0.541667,0.666667,Y,0,N,0
8,27406,F,45,160,70,3.708333,0.041667,0.005556,0.041667,0.041667,...,0.584722,0.041667,0.004167,1.000000,0.708333,0.916667,Y,0,Y,0
9,27417,M,40,180,90,3.666667,0.043056,0.041667,0.041667,0.041667,...,0.668750,0.041667,0.041667,1.500000,1.875000,1.750000,Y,0,Y,1


In [7]:
# Guardar
output_path = '../data/external/predicciones_finales.csv'
df_resultado.to_csv(output_path, index=False)

print(f'✓ Predicciones exportadas a: {output_path}')
print(f'✓ Shape: {df_resultado.shape}')
print(f'✓ Columna smoking_prediction: valores 0 y 1 (enteros)')

✓ Predicciones exportadas a: ../data/external/predicciones_finales.csv
✓ Shape: (5692, 27)
✓ Columna smoking_prediction: valores 0 y 1 (enteros)


## 5. Verificación final

In [8]:
# Leer el archivo exportado para verificar
df_check = pd.read_csv(output_path)

print('=== VERIFICACIÓN DEL ARCHIVO EXPORTADO ===')
print('Shape:', df_check.shape)
print('Columnas:', df_check.columns.tolist())
print('Nulos en smoking_prediction:', df_check['smoking_prediction'].isnull().sum())
print('Valores únicos:', sorted(df_check['smoking_prediction'].unique()))
print()
print('Distribución de predicciones:')
print(df_check['smoking_prediction'].value_counts())
print()
print(df_check[['ID', 'smoking_prediction']].head(10))

=== VERIFICACIÓN DEL ARCHIVO EXPORTADO ===
Shape: (5692, 27)
Columnas: ['ID', 'gender', 'age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'oral', 'dental caries', 'tartar', 'smoking_prediction']
Nulos en smoking_prediction: 0
Valores únicos: [np.int64(0), np.int64(1)]

Distribución de predicciones:
smoking_prediction
0    2866
1    2826
Name: count, dtype: int64

      ID  smoking_prediction
0  27358                   1
1  27364                   1
2  27368                   1
3  27378                   0
4  27381                   1
5  27386                   1
6  27388                   0
7  27393                   0
8  27406                   0
9  27417                   1
